<a href="https://colab.research.google.com/github/sebastiangome/latam-agente/blob/main/EP3_Observabilidad_LATAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente de Cuenta LATAM Airlines — Observabilidad (ISY0101, EP3)

**Evaluación Parcial N°3 — Implementación de Observabilidad | Ingeniería de Soluciones con IA, Duoc UC**

**Integrantes:** Sebastián Gomez · Rafael Pacheco

Tercera parte del proyecto. El agente es **el mismo del EP2** (módulos 1 a 6 copiados sin cambios de `EP2_Agente_LATAM.ipynb`). Este notebook le agrega:

- **Módulo 7 · Instrumentación:** registra cada llamada al LLM, cada herramienta y cada request HTTP (latencia, tokens, errores, reintentos 429).
- **Módulo 8 · Batería de pruebas:** 10 casos con variabilidad de datos (texto sin tildes, casos borde, errores del usuario, multi-intención, memoria), repetidos para medir consistencia.
- **Módulo 9 · Métricas:** precisión, consistencia, frecuencia de errores, latencia y uso de recursos.
- **Módulo 10 · Exportar logs** para el dashboard (`dashboard/app.py`, Streamlit).

> **Credenciales:** `LLM_API_KEY` (Groq) y `GOOGLE_API_KEY` (Gemini) como Secrets de Colab con *Notebook access* activado.

---

### Declaración de Uso Ético de IA

En este trabajo el equipo usó **Claude (Anthropic)** como asistente de IA para:

- Diseñar e implementar el código de observabilidad: el callback `Observador`, el conteo de requests HTTP, la función `atender_observado()`, la batería de 10 casos y el cálculo de métricas.
- Construir el dashboard en Streamlit (`dashboard/app.py`).
- Calcular las métricas a partir de los logs y redactar borradores del informe y de la presentación.

**Validación del equipo:**

- El equipo ejecutó la batería completa en Google Colab con sus propias credenciales de Groq y Gemini; todos los resultados del informe provienen de esos logs reales (`logs/`).
- Al revisar las respuestas registradas en el log, se detectó que el evaluador marcaba fallas falsas (no reconocía el guion tipográfico de "CASO‑0001" y evaluaba la respuesta truncada). Se corrigió y se reevaluó la misma ejecución; el registro original se conserva en `logs/turnos_original.jsonl`.
- El equipo revisó el código, verificó que el dashboard mostrara los datos reales y revisó y ajustó los textos del informe y la presentación. [AGREGAR: qué más revisaron o cambiaron ustedes]

Según la normativa de uso de IA de Duoc UC: https://bibliotecas.duoc.cl/ia

## MÓDULO 1 — Setup y modelos

In [ ]:
# Instalación de dependencias (Google Colab)
!pip install -q langchain langgraph langchain-groq langchain-google-genai langchain-community faiss-cpu python-dotenv psutil

In [ ]:
import os, re, json, sqlite3, unicodedata
from datetime import datetime
import numpy as np
import pandas as pd

# Credenciales: Colab Secrets o archivo .env (igual que en el EP1)
try:
    from google.colab import userdata
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY"):
        if os.getenv(_k): continue
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            print(f"[AVISO] Falta el Secret '{_k}': créalo en Secrets y activa 'Notebook access'")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

print("LLM_API_KEY (Groq):", "OK" if os.getenv("LLM_API_KEY") else "FALTA")
print("GOOGLE_API_KEY (Gemini):", "OK" if os.getenv("GOOGLE_API_KEY") else "FALTA")

In [ ]:
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage

# LLM: se prueba cada modelo en orden y se usa el primero disponible.
# En el EP1 Groq dio de baja llama-3.3-70b durante el proyecto; por eso hay respaldo.
MODELOS_GROQ = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.6-27b"]
llm = None
for _modelo in MODELOS_GROQ:
    try:
        _cand = ChatGroq(model=_modelo, api_key=os.getenv("LLM_API_KEY"), temperature=0.2, max_tokens=2000,
                         max_retries=6)  # reintenta si Groq responde 429 (límite de tokens por minuto)
        _cand.invoke([HumanMessage(content="ok")])
        llm, modelo_llm = _cand, _modelo
        print(f"[OK] LLM: {_modelo}")
        break
    except Exception as e:
        print(f"[INFO] {_modelo} no disponible: {str(e)[:80]}")
if llm is None:
    raise RuntimeError("No hay modelos Groq disponibles. Revisa LLM_API_KEY.")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001",
                                          google_api_key=os.getenv("GOOGLE_API_KEY"))
print("[OK] Embeddings: gemini-embedding-001")

## MÓDULO 2 — Datos del EP1: reservas (fuente interna) y políticas (fuente externa)

In [ ]:
# Reservas simuladas del EP1 (200 filas, semilla 42)
RUTA_CSV = "data/reservas_latam.csv"
if not os.path.exists(RUTA_CSV):
    RUTA_CSV = "https://raw.githubusercontent.com/sebastiangome/latam-agente/main/data/reservas_latam.csv"
df_reservas = pd.read_csv(RUTA_CSV)
RESERVAS = {fila["codigo_reserva"]: fila for fila in df_reservas.to_dict(orient="records")}
print(f"[OK] {len(RESERVAS)} reservas cargadas")
df_reservas.head(3)

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# Los mismos 5 resúmenes de políticas oficiales del EP1, con su URL
POLITICAS = [
    {"id": "ext_cambios_01", "tema": "cambios",
     "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/cambios/pasajes/cambiar-vuelo-fecha-pasaje",
     "texto": "Un pasajero puede cambiar la fecha o el vuelo de su pasaje siempre que las condiciones de su tarifa lo permitan. El cambio debe solicitarse antes de la salida del vuelo original; en algunos casos tambien es posible cambiarlo despues de iniciado el viaje si la tarifa lo autoriza. El proceso se realiza desde la seccion 'Mis Viajes', ingresando el codigo de reserva y el apellido del pasajero."},
    {"id": "ext_cambios_02", "tema": "cambios",
     "fuente_url": "https://www.latamairlines.com/cl/es/centro-ayuda/preguntas/problemas-vuelo/cambio-itinerario/vuelo-adelantado",
     "texto": "Si LATAM adelanta un vuelo 16 minutos o mas y el pasajero no esta conforme con el nuevo horario, puede cambiar la hora o fecha del vuelo sin costo, o solicitar la devolucion del pasaje, siempre que mantenga el mismo destino y cabina del vuelo original. El plazo para solicitar este cambio es de hasta 12 meses desde la fecha del vuelo original comprado."},
    {"id": "ext_equipaje_01", "tema": "equipaje",
     "fuente_url": "https://www.latamairlines.com/us/es/centro-ayuda/preguntas/equipaje",
     "texto": "La franquicia de equipaje facturado depende de la tarifa, cabina y ruta del pasaje. En general, la cabina Economy permite una cantidad de piezas menor que Premium Economy o Premium Business. Ademas del equipaje facturado, se permite equipaje de mano y un articulo personal, con limites de peso y tamano que tambien varian segun la tarifa contratada. Es importante verificar la franquicia especifica en el sitio oficial antes de viajar, ya que estas politicas pueden cambiar (por ejemplo, LATAM ha reducido en el pasado la cantidad de piezas gratuitas en algunos vuelos domesticos)."},
    {"id": "ext_millas_01", "tema": "millas",
     "fuente_url": "https://www.latamairlines.com/es/es/centro-ayuda/preguntas/latam-pass/millas/como-acumular",
     "texto": "Los pasajeros acumulan Millas LATAM Pass al volar con LATAM o con aerolineas asociadas, asi como en comercios asociados o mediante tarjetas de credito con convenio. Los pasajes que fueron pagados totalmente con millas no generan acumulacion de nuevas millas ni de Puntos Calificables."},
    {"id": "ext_millas_02", "tema": "millas",
     "fuente_url": "https://latampass.latam.com/es_cl/reglamento-2025/acumulacion-y-canje",
     "texto": "El canje de equipaje adicional con Millas LATAM Pass solo puede realizarse dentro del mismo flujo de canje del pasaje; no es posible canjear equipaje antes o despues de haber canjeado el pasaje asociado. La cantidad de millas necesarias para un canje depende del destino y la cabina seleccionada."},
]

indice_politicas = FAISS.from_documents(
    [Document(page_content=p["texto"], metadata={k: p[k] for k in ("id", "tema", "fuente_url")}) for p in POLITICAS],
    embeddings,
)
print(f"[OK] Índice FAISS de políticas: {indice_politicas.index.ntotal} vectores")

## MÓDULO 3 — Memoria

- **Corto plazo:** el historial de cada conversación lo guarda el *checkpointer* de LangGraph (`InMemorySaver`), identificado por `thread_id`. Así el agente entiende un "es ACF657" como respuesta a la pregunta anterior.
- **Largo plazo:** una base SQLite guarda cada atención (resumen + vector de embedding) y los casos derivados. Persiste entre conversaciones.
- **Recuperación semántica:** se filtra primero por código de reserva y luego se ordenan las atenciones por similitud coseno con la consulta actual. Nunca se mezcla la historia de dos pasajeros (mismo criterio de privacidad del EP1).

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memoria_corto_plazo = InMemorySaver()

# True: la demo parte con la memoria de largo plazo vacía (evita casos de ensayos anteriores).
# False: se conserva lo guardado en ejecuciones previas.
REINICIAR_MEMORIA = True
if REINICIAR_MEMORIA and os.path.exists("memoria_latam.db"):
    os.remove("memoria_latam.db")

db = sqlite3.connect("memoria_latam.db", check_same_thread=False)
db.executescript('''
CREATE TABLE IF NOT EXISTS atenciones (pnr TEXT, fecha TEXT, resumen TEXT, vector TEXT);
CREATE TABLE IF NOT EXISTS casos (id TEXT, pnr TEXT, fecha TEXT, tipo TEXT, prioridad TEXT, detalle TEXT, estado TEXT);
''')

def guardar_atencion(pnr, resumen):
    vector = embeddings.embed_query(resumen)
    db.execute("INSERT INTO atenciones VALUES (?, ?, ?, ?)",
               (pnr, datetime.now().isoformat(timespec="seconds"), resumen, json.dumps(vector)))
    db.commit()

def buscar_atenciones(pnr, consulta, k=2):
    filas = db.execute("SELECT fecha, resumen, vector FROM atenciones WHERE pnr = ?", (pnr,)).fetchall()
    if not filas:
        return []
    q = np.array(embeddings.embed_query(consulta))
    m = np.array([json.loads(v) for _, _, v in filas])
    sim = m @ q / (np.linalg.norm(m, axis=1) * np.linalg.norm(q))
    orden = np.argsort(-sim)[:k]
    return [{"fecha": filas[i][0], "resumen": filas[i][1], "similitud": round(float(sim[i]), 3)} for i in orden]

def casos_abiertos(pnr):
    filas = db.execute("SELECT id, tipo, prioridad FROM casos WHERE pnr = ? AND estado = 'abierto'", (pnr,)).fetchall()
    return [{"id": i, "tipo": t, "prioridad": p} for i, t, p in filas]

print("[OK] Memoria de corto plazo (checkpointer) y largo plazo (SQLite) listas")

## MÓDULO 4 — Herramientas del agente

| Tipo | Herramienta | Qué hace |
|---|---|---|
| Consulta | `consultar_reserva` | Búsqueda **exacta** del código de reserva (como en el EP1) |
| Consulta | `buscar_politicas` | RAG sobre las políticas, filtrando por tema, con URL de la fuente |
| Consulta | `recordar_historial` | Memoria de largo plazo: atenciones parecidas y casos abiertos del pasajero |
| Razonamiento | `evaluar_vuelo_adelantado` | Regla oficial: ≥ 16 min de adelanto y ≤ 12 meses → cambio sin costo o devolución |
| Escritura | `registrar_caso` | Crea un caso para un ejecutivo (el agente no ejecuta transacciones) |

La regla de vuelo adelantado se evalúa en código porque en el EP1 el LLM llegó a dar cifras que no estaban en la política recuperada ("2 piezas de hasta 32 kg").

In [ ]:
from langchain_core.tools import tool

PATRON_PNR = re.compile(r"\b[A-Z]{3}\d{3}\b")

@tool
def consultar_reserva(codigo_reserva: str) -> str:
    """Busca una reserva por su código exacto (3 letras + 3 números, ej. RAG276)."""
    fila = RESERVAS.get(codigo_reserva.strip().upper())
    if fila is None:
        return json.dumps({"encontrada": False, "codigo_reserva": codigo_reserva})
    return json.dumps({"encontrada": True, **{k: (int(v) if k == "millas_acumuladas" else v) for k, v in fila.items()}},
                      ensure_ascii=False)

@tool
def buscar_politicas(consulta: str, tema: str = "") -> str:
    """Recupera las políticas oficiales de LATAM más relevantes. tema: cambios, equipaje o millas."""
    filtro = {"tema": tema} if tema in ("cambios", "equipaje", "millas") else None
    docs = indice_politicas.similarity_search(consulta, k=2, filter=filtro)
    return json.dumps([{"texto": d.page_content, "fuente_url": d.metadata["fuente_url"]} for d in docs],
                      ensure_ascii=False)

@tool
def recordar_historial(codigo_reserva: str, consulta: str) -> str:
    """Recupera atenciones anteriores parecidas y casos abiertos del pasajero (memoria de largo plazo)."""
    pnr = codigo_reserva.strip().upper()
    if pnr not in RESERVAS:
        return json.dumps({"error": "reserva no verificada"})
    return json.dumps({"atenciones_previas": buscar_atenciones(pnr, consulta),
                       "casos_abiertos": casos_abiertos(pnr)}, ensure_ascii=False)

@tool
def evaluar_vuelo_adelantado(minutos_adelanto: int, meses_desde_vuelo: float = 0) -> str:
    """Aplica la regla oficial: si el vuelo se adelantó 16 minutos o más y la solicitud es dentro
    de 12 meses, el pasajero puede cambiar sin costo o pedir la devolución."""
    elegible = minutos_adelanto >= 16 and meses_desde_vuelo <= 12
    if minutos_adelanto < 16:
        motivo = f"El adelanto ({minutos_adelanto} min) es menor a 16 minutos: no aplica el cambio sin costo."
    elif meses_desde_vuelo > 12:
        motivo = "La solicitud está fuera del plazo de 12 meses."
    else:
        motivo = "Elegible: puede cambiar hora o fecha sin costo, o pedir devolución, con el mismo destino y cabina."
    return json.dumps({"elegible": elegible, "motivo": motivo, "fuente_url": POLITICAS[1]["fuente_url"]},
                      ensure_ascii=False)

@tool
def evalua_vuelo_adelantado(minutos_adelanto: int, meses_desde_vuelo: float = 0) -> str:
    """Alias compatible para la herramienta evaluar_vuelo_adelantado."""
    return evaluar_vuelo_adelantado.invoke({"minutos_adelanto": minutos_adelanto,
                                             "meses_desde_vuelo": meses_desde_vuelo})

@tool
def registrar_caso(tipo: str, prioridad: str, detalle: str, codigo_reserva: str = "") -> str:
    """Registra un caso para que lo gestione un ejecutivo humano. El código de reserva es opcional. tipo: vuelo_adelantado | transaccion | fuera_de_cobertura. prioridad: alta | media | baja."""
    n = db.execute("SELECT COUNT(*) FROM casos").fetchone()[0] + 1
    caso_id = f"CASO-{n:04d}"
    db.execute("INSERT INTO casos VALUES (?, ?, ?, ?, ?, ?, 'abierto')",
               (caso_id, codigo_reserva.upper(), datetime.now().isoformat(timespec="seconds"), tipo, prioridad, detalle))
    db.commit()
    return json.dumps({"caso_id": caso_id, "prioridad": prioridad, "estado": "abierto"})

HERRAMIENTAS = [consultar_reserva, buscar_politicas, recordar_historial, evaluar_vuelo_adelantado,
                evalua_vuelo_adelantado, registrar_caso]
print("[OK] Herramientas:", [t.name for t in HERRAMIENTAS])

## MÓDULO 5 — Planificación con prioridades

Antes de **cada** llamada al modelo, `planificar()` arma el plan del turno a partir de la consulta y de lo que ya devolvieron las herramientas, y lo agrega al prompt de sistema. Así el agente:

1. **Ordena** los temas por prioridad: vuelo adelantado y transacciones (alta) → cambios y equipaje (media) → millas (baja).
2. **Verifica primero** la reserva (privacidad) y, si no hay código, lo pide.
3. **Replanifica** cuando cambian las condiciones: reserva inexistente, vuelo elegible a cambio sin costo, o caso ya abierto en la memoria de largo plazo.

In [ ]:
TEMAS = {  # tema: (palabras clave, prioridad)
    "vuelo_adelantado": (r"adelant", 1),
    "transaccion": (r"cambiame|canjeame|cancela|reembols|devuelveme|hazme el cambio", 1),
    "cambios": (r"cambi|modific|otra fecha", 2),
    "equipaje": (r"equipaje|maleta|pieza|kg|kilo", 2),
    "millas": (r"milla|latam pass|acumul|canje", 3),
}

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in texto if not unicodedata.combining(c))

def planificar(consulta, resultados=None):
    """Devuelve el plan del turno. resultados = {herramienta: salida} ya obtenidos en este turno."""
    resultados = resultados or {}
    t = normalizar(consulta)
    temas = sorted([tm for tm, (patron, _) in TEMAS.items() if re.search(patron, t)], key=lambda tm: TEMAS[tm][1])
    if "vuelo_adelantado" in temas and "cambios" in temas:
        temas.remove("cambios")
    if "transaccion" in temas and "cambios" in temas:
        temas.remove("cambios")
    pnr_m = PATRON_PNR.search(consulta.upper())
    pnr = pnr_m.group(0) if pnr_m else None
    minutos = re.search(r"(\d+)\s*min", t)
    pasos, decisiones = [], []

    if not temas:
        pasos.append("No hay política para este tema: NO respondas con conocimiento propio. registrar_caso(tipo='fuera_de_cobertura', prioridad='media') sin pedir el código de reserva, e informa al pasajero el ID del caso.")
        return {"temas": ["fuera_de_cobertura"], "pnr": pnr, "pasos": pasos, "decisiones": decisiones}

    # 1) Identidad / privacidad
    reserva = resultados.get("consultar_reserva")
    if pnr is None:
        pasos.append("Sin código de reserva: responde solo la política general y PIDE el código.")
    elif reserva is not None and not reserva.get("encontrada"):
        decisiones.append(f"Replanificación: la reserva {pnr} no existe → no entregar datos personales.")
        pasos.append("Pide al pasajero revisar el código. No uses recordar_historial ni registrar_caso.")
    else:
        pasos.append(f"consultar_reserva('{pnr}')")
        pasos.append(f"recordar_historial('{pnr}', consulta)")

    # 2) Temas en orden de prioridad
    historial = resultados.get("recordar_historial") or {}
    caso_previo = [c for c in historial.get("casos_abiertos", []) if c["tipo"] == "vuelo_adelantado"]
    for tema in temas:
        if tema == "vuelo_adelantado":
            if caso_previo:
                decisiones.append(f"Memoria de largo plazo: ya existe {caso_previo[0]['id']} → informar estado, no duplicar.")
                continue
            pasos.append("buscar_politicas(consulta, tema='cambios')")
            if minutos:
                pasos.append(f"evaluar_vuelo_adelantado(minutos_adelanto={minutos.group(1)})")
            else:
                pasos.append("Pide cuántos minutos se adelantó el vuelo.")
            evaluacion = resultados.get("evaluar_vuelo_adelantado")
            if evaluacion and evaluacion.get("elegible"):
                decisiones.append("Replanificación: vuelo elegible → registrar caso de prioridad ALTA.")
                pasos.append("registrar_caso(tipo='vuelo_adelantado', prioridad='alta')")
        elif tema == "transaccion":
            pasos.append("El agente NO ejecuta transacciones: registrar_caso(tipo='transaccion', prioridad='alta')")
        else:
            pasos.append(f"buscar_politicas(consulta, tema='{tema}')")
    pasos.append("Responde citando la fuente_url. No des cifras que no vengan de una herramienta.")
    return {"temas": temas, "pnr": pnr, "pasos": pasos, "decisiones": decisiones}

planificar("Reserva CKJ320: me adelantaron el vuelo 40 minutos y quiero saber cómo acumulo millas")

In [ ]:
# Pruebas del planificador (sin LLM)
p = planificar("¿Puedo cambiar mi vuelo?")
assert p["pnr"] is None and "PIDE el código" in p["pasos"][0]

p = planificar("Reserva RAG276: ¿cómo acumulo millas? Me adelantaron el vuelo 30 minutos")
assert p["temas"] == ["vuelo_adelantado", "millas"]          # prioridad alta antes que baja

p = planificar("Reserva ZZZ999, ¿equipaje?", {"consultar_reserva": {"encontrada": False}})
assert any("no existe" in d for d in p["decisiones"])         # replanifica: reserva inexistente

p = planificar("Reserva CKJ320, adelantaron 40 minutos", {"evaluar_vuelo_adelantado": {"elegible": True}})
assert any("registrar_caso" in s for s in p["pasos"])         # replanifica: vuelo elegible

p = planificar("Reserva QVA863, adelantaron 10 minutos", {"evaluar_vuelo_adelantado": {"elegible": False}})
assert not any("registrar_caso" in s for s in p["pasos"])     # no aplica: sin caso

p = planificar("¿Puedo viajar con mi perro?")
assert p["temas"] == ["fuera_de_cobertura"]

assert json.loads(evaluar_vuelo_adelantado.invoke({"minutos_adelanto": 15}))["elegible"] is False
assert json.loads(evaluar_vuelo_adelantado.invoke({"minutos_adelanto": 16}))["elegible"] is True
assert json.loads(evalua_vuelo_adelantado.invoke({"minutos_adelanto": 16}))["elegible"] is True
assert json.loads(consultar_reserva.invoke({"codigo_reserva": "ZZZ999"}))["encontrada"] is False
print("[OK] 10 pruebas del planificador y las herramientas aprobadas")

## MÓDULO 6 — Agente (`create_agent` de LangChain v1)

El agente es el bucle ReAct de LangGraph: el modelo decide qué herramienta llamar, ve el resultado y vuelve a decidir. Un *middleware* (`prompt_con_plan`) recalcula el plan antes de cada llamada al modelo y lo agrega al `SYSTEM_PROMPT` del EP1.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt
from langchain_core.messages import HumanMessage, ToolMessage

SYSTEM_PROMPT = """Eres un asistente de atencion al pasajero para LATAM Airlines.
Tu funcion es responder consultas sobre cambios de vuelo, equipaje y Millas LATAM Pass.

Reglas:
- Usa SOLO la informacion que entregan las herramientas (reserva del pasajero y politicas
  oficiales). No inventes datos ni reglas.
- Usa los nombres de herramientas exactamente como están definidos, especialmente evaluar_vuelo_adelantado.
- Si el pasajero indica los minutos de adelanto, evalúa primero la regla con evaluar_vuelo_adelantado; sigue el plan y registra el caso solo si la evaluación lo indica.
- No afirmes costos, cargos, piezas, kilos ni condiciones de una tarifa que no estén en la
  política recuperada: si dependen de la tarifa, dilo así y remite a latamairlines.com.
- Si la consulta depende de un dato que no tienes, pide ese dato al pasajero en vez de asumirlo.
- Las politicas de LATAM pueden cambiar: recomienda verificar la vigencia en latamairlines.com.
- No ejecutas transacciones reales (cambios, canjes); solo informas, orientas y registras casos.
- Responde en español, en un tono claro y cercano para un pasajero."""

def turno_actual(mensajes):
    """Consulta del pasajero (une la anterior si la actual solo trae un dato) y resultados del turno."""
    humanos = [i for i, m in enumerate(mensajes) if m.type == "human"]
    consulta = mensajes[humanos[-1]].content
    solo_un_dato = len(consulta.split()) <= 4 and planificar(consulta)["temas"] == ["fuera_de_cobertura"]
    if len(humanos) > 1 and solo_un_dato:   # continuidad: "es ACF657" responde a la pregunta anterior
        consulta = mensajes[humanos[-2]].content + " " + consulta
    resultados = {}
    for m in mensajes[humanos[-1]:]:
        if isinstance(m, ToolMessage):
            try:
                nombre = "evaluar_vuelo_adelantado" if m.name == "evalua_vuelo_adelantado" else m.name
                resultados[nombre] = json.loads(m.content)
            except json.JSONDecodeError:
                pass
    return consulta, resultados

@dynamic_prompt
def prompt_con_plan(request):
    consulta, resultados = turno_actual(request.state["messages"])
    plan = planificar(consulta, resultados)
    texto_plan = "\n".join(f"{i}. {p}" for i, p in enumerate(plan["pasos"], 1))
    decisiones = "\n".join(f"- {d}" for d in plan["decisiones"]) or "- ninguna"
    return f"{SYSTEM_PROMPT}\n\nPLAN (ordenado por prioridad, temas: {plan['temas']}):\n{texto_plan}\n\nDECISIONES:\n{decisiones}"

agente = create_agent(model=llm, tools=HERRAMIENTAS, middleware=[prompt_con_plan],
                      checkpointer=memoria_corto_plazo)

def atender(consulta, sesion):
    """Un turno de atención. Muestra plan, herramientas usadas y respuesta, y guarda la atención."""
    estado = agente.invoke({"messages": [HumanMessage(content=consulta)]},
                           config={"configurable": {"thread_id": sesion}})
    mensajes = estado["messages"]
    c, resultados = turno_actual(mensajes)
    plan = planificar(c, resultados)
    usadas = [m.name for m in mensajes[[i for i, m in enumerate(mensajes) if m.type == "human"][-1]:]
              if isinstance(m, ToolMessage)]
    respuesta = mensajes[-1].content
    print(f"PASAJERO: {consulta}\nTEMAS: {plan['temas']}")
    for d in plan["decisiones"]:
        print("DECISIÓN:", d)
    print("HERRAMIENTAS:", " → ".join(usadas) or "ninguna")
    print("\nAGENTE:", respuesta)
    reserva = resultados.get("consultar_reserva", {})
    if reserva.get("encontrada"):   # memoria de largo plazo solo para reservas verificadas
        guardar_atencion(reserva["codigo_reserva"], f"Consulta: {c[:200]} | Temas: {plan['temas']} | "
                                                    f"Respuesta: {respuesta[:200]}")
    return respuesta

## MÓDULO 7 — Instrumentación (observabilidad)

Se registran tres niveles, sin modificar el agente:

| Nivel | Cómo se captura | Qué se mide |
|---|---|---|
| **Evento** (cada llamada al LLM y a cada herramienta) | `Observador`, un *callback* de LangChain que se pasa en `agente.invoke(config={"callbacks": [...]})` | latencia, tokens de entrada y salida, herramienta elegida, errores |
| **HTTP** (cada request a Groq y Gemini) | un `logging.Handler` sobre el logger de `httpx` | código de estado; cuenta los **429** (límite de tokens por minuto) aunque el cliente los reintente solo |
| **Turno** (una consulta completa) | `atender_observado()` | latencia total, herramientas en orden, plan, decisión correcta o no, memoria RAM, tamaño de la base |

Salidas en la carpeta `logs/`:
- `eventos.jsonl` — una línea por llamada al LLM, herramienta o request HTTP.
- `turnos.jsonl` — una línea por consulta, con las métricas del turno.
- `agente.log` — registro de ejecución en texto, con niveles INFO / WARNING / ERROR.

In [ ]:
import time, uuid, logging, psutil
from pathlib import Path
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import HumanMessage, ToolMessage
from IPython.display import display

DIR_LOGS = Path("logs"); DIR_LOGS.mkdir(exist_ok=True)
F_EVENTOS, F_TURNOS = DIR_LOGS / "eventos.jsonl", DIR_LOGS / "turnos.jsonl"

# Registro de ejecución en texto (agente.log)
log = logging.getLogger("agente_latam")
log.setLevel(logging.INFO); log.propagate = False
if not log.handlers:
    _fh = logging.FileHandler(DIR_LOGS / "agente.log", encoding="utf-8")
    _fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    log.addHandler(_fh)

TURNO = {"id": None}          # turno en curso: los eventos se asocian a él
ALIAS = {"evalua_vuelo_adelantado": "evaluar_vuelo_adelantado"}

def escribir(archivo, registro):
    with open(archivo, "a", encoding="utf-8") as f:
        f.write(json.dumps(registro, ensure_ascii=False, default=str) + "\n")

def evento(**datos):
    escribir(F_EVENTOS, {"ts": datetime.now().isoformat(timespec="milliseconds"), "turno_id": TURNO["id"], **datos})

class Observador(BaseCallbackHandler):
    """Callback de LangChain: mide cada llamada al LLM y cada herramienta del turno."""
    def __init__(self):
        self.t0, self.nombre = {}, {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kw):
        self.t0[run_id] = time.perf_counter()

    def on_llm_end(self, response, *, run_id, **kw):
        dt = time.perf_counter() - self.t0.pop(run_id, time.perf_counter())
        msg = response.generations[0][0].message
        uso = getattr(msg, "usage_metadata", None) or {}
        llamadas = [t["name"] for t in getattr(msg, "tool_calls", [])]
        evento(tipo="llm", nombre=(msg.response_metadata or {}).get("model_name", "llm"), latencia_s=round(dt, 3),
               tokens_in=uso.get("input_tokens", 0), tokens_out=uso.get("output_tokens", 0),
               decision=",".join(llamadas) or "respuesta_final")
        log.info(f"turno={TURNO['id']} LLM {dt:.2f}s in={uso.get('input_tokens',0)} out={uso.get('output_tokens',0)} -> {','.join(llamadas) or 'respuesta_final'}")

    def on_llm_error(self, error, *, run_id, **kw):
        dt = time.perf_counter() - self.t0.pop(run_id, time.perf_counter())
        evento(tipo="llm", nombre="llm", latencia_s=round(dt, 3), error=type(error).__name__, detalle=str(error)[:300])
        log.error(f"turno={TURNO['id']} LLM ERROR {type(error).__name__}: {str(error)[:200]}")

    def on_tool_start(self, serialized, input_str, *, run_id, **kw):
        self.t0[run_id] = time.perf_counter()
        self.nombre[run_id] = (serialized or {}).get("name", "herramienta")

    def on_tool_end(self, output, *, run_id, **kw):
        dt = time.perf_counter() - self.t0.pop(run_id, time.perf_counter())
        nombre = self.nombre.pop(run_id, "herramienta")
        evento(tipo="herramienta", nombre=ALIAS.get(nombre, nombre), alias=nombre in ALIAS, latencia_s=round(dt, 3))
        nivel = log.warning if nombre in ALIAS else log.info
        nivel(f"turno={TURNO['id']} TOOL {nombre} {dt:.3f}s" + (" (alias: el modelo escribió mal el nombre)" if nombre in ALIAS else ""))

    def on_tool_error(self, error, *, run_id, **kw):
        dt = time.perf_counter() - self.t0.pop(run_id, time.perf_counter())
        nombre = self.nombre.pop(run_id, "herramienta")
        evento(tipo="herramienta", nombre=ALIAS.get(nombre, nombre), latencia_s=round(dt, 3), error=type(error).__name__)
        log.error(f"turno={TURNO['id']} TOOL {nombre} ERROR {type(error).__name__}: {str(error)[:200]}")

class ContadorHTTP(logging.Handler):
    """Lee el log de httpx: cada request a Groq/Gemini con su código de estado (incluye reintentos)."""
    PATRON = re.compile(r'(?:POST|GET) https?://([^/\s]+)\S* "HTTP/[\d.]+ (\d{3})')
    def emit(self, record):
        m = self.PATRON.search(record.getMessage())
        if not m:
            return
        host, estado = m.group(1), int(m.group(2))
        evento(tipo="http", nombre=host, estado=estado)
        if estado == 429:
            log.warning(f"turno={TURNO['id']} HTTP 429 {host}: límite de tokens por minuto, el cliente reintenta")
        elif estado >= 400:
            log.error(f"turno={TURNO['id']} HTTP {estado} {host}")

_httpx = logging.getLogger("httpx")
_httpx.setLevel(logging.INFO); _httpx.propagate = False
if not any(isinstance(h, ContadorHTTP) for h in _httpx.handlers):
    _httpx.addHandler(ContadorHTTP())

print("[OK] Instrumentación lista. Logs en:", DIR_LOGS.resolve())

In [ ]:
def herramientas_del_turno(mensajes):
    ult = [i for i, m in enumerate(mensajes) if m.type == "human"][-1]
    return [ALIAS.get(m.name, m.name) for m in mensajes[ult:] if isinstance(m, ToolMessage)]

LIMITE_PASOS = 20   # corta un turno en bucle (C08, el más largo, usa unos 14 pasos) (queda como GraphRecursionError)

def atender_observado(consulta, sesion, caso=None, rep=0, previa=False):
    """Igual que atender() del EP2, pero mide el turno y lo registra en logs/."""
    TURNO["id"] = f"{caso or 'manual'}-r{rep}{'-previa' if previa else ''}-{uuid.uuid4().hex[:6]}"
    proceso = psutil.Process()
    log.info(f"turno={TURNO['id']} INICIO caso={caso} rep={rep} consulta={consulta!r}")
    t0 = time.perf_counter()
    registro = {"turno_id": TURNO["id"], "ts": datetime.now().isoformat(timespec="seconds"), "caso": caso,
                "rep": rep, "previa": previa, "consulta": consulta, "modelo": modelo_llm, "error": None}
    try:
        estado = agente.invoke({"messages": [HumanMessage(content=consulta)]},
                               config={"configurable": {"thread_id": sesion}, "callbacks": [Observador()],
                                       "recursion_limit": LIMITE_PASOS})
        mensajes = estado["messages"]
        c, resultados = turno_actual(mensajes)
        plan = planificar(c, resultados)
        respuesta = mensajes[-1].content or ""
        ult = [i for i, m in enumerate(mensajes) if m.type == "human"][-1]
        uso = [m.usage_metadata or {} for m in mensajes[ult:] if m.type == "ai"]
        registro.update(temas=plan["temas"], decisiones=plan["decisiones"],
                        herramientas=herramientas_del_turno(mensajes), respuesta=respuesta[:3000],
                        mensajes_historial=len(mensajes),
                        tokens_in=sum(u.get("input_tokens", 0) for u in uso),
                        tokens_out=sum(u.get("output_tokens", 0) for u in uso))
        reserva = resultados.get("consultar_reserva", {})
        if reserva.get("encontrada"):   # misma lógica de memoria de largo plazo del EP2
            guardar_atencion(reserva["codigo_reserva"], f"Consulta: {c[:200]} | Temas: {plan['temas']} | Respuesta: {respuesta[:200]}")
    except Exception as e:
        registro.update(error=type(e).__name__, detalle_error=str(e)[:300], herramientas=[], temas=[], respuesta="")
        log.error(f"turno={TURNO['id']} TURNO FALLIDO {type(e).__name__}: {str(e)[:200]}")
    registro["latencia_s"] = round(time.perf_counter() - t0, 3)
    registro["rss_mb"] = round(proceso.memory_info().rss / 1e6, 1)
    registro["db_kb"] = round(os.path.getsize("memoria_latam.db") / 1024, 1) if os.path.exists("memoria_latam.db") else 0
    log.info(f"turno={TURNO['id']} FIN {registro['latencia_s']}s herramientas={registro.get('herramientas')} error={registro['error']}")
    TURNO["id"] = None
    return registro

## MÓDULO 8 — Batería de pruebas con variabilidad de datos

Cada caso define qué **debe** hacer el agente (temas que debe detectar el planificador, herramientas que debe usar, herramientas que **no** debe usar y un patrón esperado o prohibido en la respuesta). Con eso se mide la **precisión**. Cada caso se repite `REPETICIONES` veces en conversaciones nuevas para medir la **consistencia**.

Antes de cada turno se vacía la memoria de largo plazo, para que una repetición no afecte a la siguiente. El caso `C10` primero ejecuta una consulta previa (no se cuenta en las métricas) para que exista un caso abierto.

> **Costo:** cada turno hace de 2 a 6 llamadas al LLM. Con 10 casos × 2 repeticiones son unos 22 turnos. Con el plan gratuito de Groq (8.000 tokens por minuto en `gpt-oss-120b`) la batería tarda varios minutos y aparecerán reintentos 429: **eso también se registra y es parte del análisis**. La batería es reanudable: si se corta, al volver a ejecutar salta los turnos ya registrados.

In [ ]:
CASOS = [
  {"id": "C01", "variante": "base", "consulta": "Tengo la reserva RAG276, ¿cuánto equipaje puedo llevar?",
   "temas": ["equipaje"], "debe": ["consultar_reserva", "buscar_politicas"], "no_debe": ["registrar_caso"],
   "no_patron": r"\b\d+\s*(kg|kilos)\b"},
  {"id": "C02", "variante": "texto sin tildes", "consulta": "hola mi reserva es rag276 cuanto equipaje llevo",
   "temas": ["equipaje"], "debe": ["consultar_reserva", "buscar_politicas"], "no_debe": ["registrar_caso"],
   "no_patron": r"\b\d+\s*(kg|kilos)\b"},
  {"id": "C03", "variante": "regla: no aplica", "consulta": "Reserva QVA863: me adelantaron el vuelo 10 minutos, ¿lo puedo cambiar gratis?",
   "temas": ["vuelo_adelantado"], "debe": ["consultar_reserva", "evaluar_vuelo_adelantado"], "no_debe": ["registrar_caso"]},
  {"id": "C04", "variante": "regla: caso borde 16 min", "consulta": "Reserva QVA863: LATAM adelantó mi vuelo 16 minutos, ¿qué puedo hacer?",
   "temas": ["vuelo_adelantado"], "debe": ["evaluar_vuelo_adelantado", "registrar_caso"], "no_debe": [],
   "patron": r"CASO-\d+"},
  {"id": "C05", "variante": "regla: aplica", "consulta": "Reserva CKJ320: LATAM adelantó 40 minutos el vuelo de mañana, ¿lo puedo cambiar gratis?",
   "temas": ["vuelo_adelantado"], "debe": ["evaluar_vuelo_adelantado", "registrar_caso"], "no_debe": [],
   "patron": r"CASO-\d+"},
  {"id": "C06", "variante": "error del usuario", "consulta": "Mi reserva es ZZZ999, ¿cuánto equipaje tengo?",
   "temas": ["equipaje"], "debe": ["consultar_reserva"], "no_debe": ["recordar_historial", "registrar_caso"]},
  {"id": "C07", "variante": "fuera de cobertura", "consulta": "¿Puedo viajar con mi perro en la cabina?",
   "temas": ["fuera_de_cobertura"], "debe": ["registrar_caso"], "no_debe": ["consultar_reserva", "buscar_politicas"],
   "patron": r"CASO-\d+"},
  {"id": "C08", "variante": "multi-intención", "consulta": "Reserva RAG276: me adelantaron el vuelo 30 minutos y además quiero saber cómo acumulo millas",
   "temas": ["vuelo_adelantado", "millas"], "debe": ["evaluar_vuelo_adelantado", "registrar_caso", "buscar_politicas"], "no_debe": []},
  {"id": "C09", "variante": "sin código de reserva", "consulta": "¿Puedo cambiar mi vuelo de mañana?",
   "temas": ["cambios"], "debe": ["buscar_politicas"], "no_debe": ["consultar_reserva", "registrar_caso"]},
  {"id": "C10", "variante": "memoria de largo plazo",
   "previa": "Reserva CKJ320: LATAM adelantó 40 minutos el vuelo de mañana, ¿lo puedo cambiar gratis?",
   "consulta": "Hola, soy de la reserva CKJ320, ¿qué pasó con lo del vuelo adelantado?",
   "temas": ["vuelo_adelantado"], "debe": ["recordar_historial"], "no_debe": ["registrar_caso"], "patron": r"CASO-\d+"},
]

def evaluar(caso, reg):
    """Compara lo que hizo el agente con lo esperado. Devuelve (ok_plan, ok_decision, motivos)."""
    usadas, motivos = set(reg.get("herramientas", [])), []
    ok_plan = reg.get("temas") == caso["temas"]
    faltan = [h for h in caso["debe"] if h not in usadas]
    sobran = [h for h in caso["no_debe"] if h in usadas]
    if faltan: motivos.append("no usó " + ", ".join(faltan))
    if sobran: motivos.append("usó " + ", ".join(sobran) + " (no debía)")
    # el modelo escribe guiones tipográficos (CASO‑0001 con U+2011): se normalizan antes de comparar
    resp = re.sub(r"[\u2010-\u2015\u2212]", "-", reg.get("respuesta", ""))
    if caso.get("patron") and not re.search(caso["patron"], resp):
        motivos.append("la respuesta no menciona el caso registrado")
    if caso.get("no_patron") and re.search(caso["no_patron"], resp, re.I):
        motivos.append("la respuesta inventa cifras que no están en la política")
    if reg.get("error"):
        motivos.append("error: " + reg["error"])
    ok_decision = not motivos            # la decisión se evalúa por lo que hizo el agente
    if not ok_plan:
        motivos.append(f"plan: temas {reg.get('temas')} en vez de {caso['temas']}")
    return ok_plan, ok_decision, motivos

def limpiar_memoria_largo_plazo():
    db.execute("DELETE FROM casos"); db.execute("DELETE FROM atenciones"); db.commit()

print(f"[OK] {len(CASOS)} casos definidos")

In [ ]:
REPETICIONES = 2      # consistencia: cada caso se repite en conversaciones nuevas
PAUSA_S = 2           # pausa entre turnos para no saturar el límite de tokens por minuto

hechos = set()
if F_TURNOS.exists():  # reanudar: no repetir turnos ya registrados con este modelo
    for linea in open(F_TURNOS, encoding="utf-8"):
        r = json.loads(linea)
        if not r.get("previa") and r.get("modelo") == modelo_llm and not r.get("error"):
            hechos.add((r["caso"], r["rep"]))

for rep in range(1, REPETICIONES + 1):
    for caso in CASOS:
        if (caso["id"], rep) in hechos:
            continue
        limpiar_memoria_largo_plazo()
        if caso.get("previa"):
            previo = atender_observado(caso["previa"], f"obs-{caso['id']}-r{rep}-previa", caso["id"], rep, previa=True)
            escribir(F_TURNOS, previo)
            time.sleep(PAUSA_S)
        reg = atender_observado(caso["consulta"], f"obs-{caso['id']}-r{rep}", caso["id"], rep)
        reg["variante"] = caso["variante"]
        reg["ok_plan"], reg["ok_decision"], reg["motivos"] = evaluar(caso, reg)
        escribir(F_TURNOS, reg)
        estado = "OK " if reg["ok_decision"] else "FALLA"
        print(f"{estado} {caso['id']} r{rep} {reg['latencia_s']:6.1f}s  {' → '.join(reg['herramientas']) or '-'}"
              + (f"  [{'; '.join(reg['motivos'])}]" if reg["motivos"] else ""))
        time.sleep(PAUSA_S)
print("[OK] Batería terminada")

## MÓDULO 9 — Métricas de observabilidad

| Métrica | Definición |
|---|---|
| **Precisión del planificador** | % de turnos donde `planificar()` detectó los temas esperados |
| **Precisión de la decisión** | % de turnos donde el agente usó las herramientas que debía, no usó las prohibidas y la respuesta cumple el patrón |
| **Consistencia** | por caso, % de repeticiones que usan el mismo conjunto de herramientas que la repetición más frecuente |
| **Frecuencia de errores** | turnos fallidos, reintentos HTTP 429 y uso del alias (nombre de herramienta mal escrito) |
| **Latencia** | p50 y p95 por turno; reparto entre LLM y herramientas |
| **Uso de recursos** | tokens de entrada y salida, llamadas al LLM por turno, memoria RAM y tamaño de la base |

In [ ]:
turnos = pd.read_json(F_TURNOS, lines=True)
turnos = turnos[(turnos["previa"] != True) & (turnos["modelo"] == modelo_llm)].copy()
turnos[["ok_plan", "ok_decision"]] = turnos[["ok_plan", "ok_decision"]].astype(bool)
eventos = pd.read_json(F_EVENTOS, lines=True)
ev = eventos[eventos["turno_id"].isin(turnos["turno_id"])]

def consistencia(g):
    firmas = g["herramientas"].apply(lambda h: tuple(sorted(set(h))))
    return firmas.value_counts().iloc[0] / len(firmas)

llm = ev[ev["tipo"] == "llm"]; tools = ev[ev["tipo"] == "herramienta"]; http = ev[ev["tipo"] == "http"]
metricas = {
    "Turnos evaluados": len(turnos),
    "Precisión planificador (%)": round(100 * turnos["ok_plan"].mean(), 1),
    "Precisión decisión (%)": round(100 * turnos["ok_decision"].mean(), 1),
    "Consistencia (%)": round(100 * turnos.groupby("caso").apply(consistencia).mean(), 1),
    "Turnos con error (%)": round(100 * turnos["error"].notna().mean(), 1),
    "Reintentos HTTP 429": int((http["estado"] == 429).sum()),
    "Uso del alias": int(tools.get("alias", pd.Series(dtype=bool)).fillna(False).sum()),
    "Latencia p50 (s)": round(turnos["latencia_s"].quantile(.5), 2),
    "Latencia p95 (s)": round(turnos["latencia_s"].quantile(.95), 2),
    "% del tiempo en el LLM": round(100 * llm["latencia_s"].sum() / max(turnos["latencia_s"].sum(), 1e-9), 1),
    "Llamadas al LLM por turno": round(len(llm) / max(len(turnos), 1), 2),
    "Tokens de entrada por turno": int(turnos["tokens_in"].mean()),
    "Tokens de salida por turno": int(turnos["tokens_out"].mean()),
    "RAM máx. (MB)": turnos["rss_mb"].max(),
}
display(pd.Series(metricas, name="valor", dtype=object).to_frame())
display(turnos.groupby(["caso", "variante"]).agg(
    precision=("ok_decision", "mean"), latencia_media_s=("latencia_s", "mean"),
    tokens_in=("tokens_in", "mean"), n=("turno_id", "count")).round(2))
display(turnos.loc[~turnos["ok_decision"], ["caso", "rep", "herramientas", "motivos"]])

## MÓDULO 10 — Exportar logs para el dashboard

Colab borra los archivos al cerrar la sesión. Esta celda descarga `logs.zip`: descomprímelo en la carpeta `logs/` del repositorio y súbelo a GitHub. El dashboard (`dashboard/app.py`) lee esos archivos.

In [ ]:
import shutil
shutil.make_archive("logs", "zip", DIR_LOGS)
try:
    from google.colab import files
    files.download("logs.zip")
except ImportError:
    print("Fuera de Colab: los logs ya están en", DIR_LOGS.resolve())